# Structural summaries: `Sequential.summary` and `Graph.summary`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jejjohnson/xr_toolz/blob/main/docs/notebooks/sequential_summary_demo.ipynb)

This notebook walks through the **shape inference** protocol in `xr_toolz` — the same idea
Keras uses for `Model.summary()`, but for `xr_toolz` pipelines.

Three pieces:

1. **`Signature`** — a tiny value object: `{dim_name: size | None} + dtype`. It carries
   *what shape an op expects/produces* without holding any data.
2. **`Operator.compute_output_signature(input_signature)`** — every operator can declare
   how it transforms shapes. The base default is shape-preserving; ops that rename, subset,
   reduce, regrid, etc. override it.
3. **`Sequential.summary(signature)` and `Graph.summary(...)`** — render an end-to-end
   table of input/output signatures per step, no real data needed.

Below: the basics, four worked pipelines on different tasks, multi-input metric graphs,
nested `Graph`-in-`Graph` composition, and a couple of common error/edge cases.


## Setup

Colab-friendly install + imports.


In [1]:
import subprocess
import sys


try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "xr_toolz"],
        check=True,
    )


In [2]:
import numpy as np
import xarray as xr

# Render xarray reprs as HTML and keep every section collapsed by default —
# readers expand only what they need; rendered notebooks stay tidy on the
# docs site.
xr.set_options(
    display_style="html",
    display_expand_data=False,
    display_expand_attrs=False,
    display_expand_coords=False,
    display_expand_data_vars=False,
)

from xr_toolz import Sequential, Signature
from xr_toolz.core import Graph, Input
from xr_toolz.geo.operators import (
    CalculateClimatology,
    Reduce,
    RenameCoords,
    SubsetBBox,
    SubsetTime,
    ValidateCoords,
)
from xr_toolz.interpolate.operators import Coarsen, RegridLike, ResampleTime
from xr_toolz.metrics.operators import RMSE

## 1. `Signature` basics

A `Signature` captures dim sizes and a dtype. `None` means "size not known
symbolically" — you'll see this on operators where the output size depends on
the data (irregular bbox crops, time-resamples, climatology grouping).

In [3]:
sig = Signature({"time": 365, "lat": 181, "lon": 360}, dtype="float32")
print(sig.format())

# Unknown sizes render as `?`:
print(Signature({"time": None, "lat": 4}).format())

(time=365, lat=181, lon=360); dtype=float32
(time=?, lat=4)


`Signature` is a value object: it's frozen, the `dims` mapping is immutable,
and equality is order-insensitive on dims plus dtype-canonicalized so string
and numpy dtype forms unify.

In [4]:
# All three of these are equal:
a = Signature({"time": 12, "lat": 4}, dtype="float32")
b = Signature({"lat": 4, "time": 12}, dtype="float32")  # different dim order
c = Signature({"time": 12, "lat": 4}, dtype=np.float32)  # numpy dtype form

print(a == b == c)
print("canonicalized dtype:", c.dtype)

True
canonicalized dtype: float32


Helpers `replace_dims`, `rename_dims`, `drop_dims` return *new* signatures
(no in-place mutation):

In [5]:
print(sig.replace_dims({"lat": None}).format())  # a regrid the size of which we don't know
print(sig.rename_dims({"lon": "longitude"}).format())  # CF rename
print(sig.drop_dims(("lat", "lon")).format())  # spatial reduction

(time=365, lat=?, lon=360); dtype=float32
(time=365, lat=181, longitude=360); dtype=float32
(time=365); dtype=float32


## 2. Sequential pipeline — SSH preprocessing

The canonical "open a CMEMS / GLORYS Dataset → validate → crop to a region →
resample time → spatially reduce" preprocessing chain. Each `summary` row
shows what shape the step receives and what it emits.

In [6]:
ssh_pipe = Sequential([
    ValidateCoords(),
    SubsetBBox(lon_bnds=(-80, -50), lat_bnds=(30, 45)),  # Gulf Stream
    ResampleTime(freq="1D", method="mean"),
    Reduce(op="mean", dim=("lat", "lon")),
])

# Start with a daily 1°×1° year-long CMEMS-like cube:
input_signature = Signature(
    {"time": 365, "lat": 181, "lon": 360}, dtype="float32",
)

print(ssh_pipe.summary(input_signature))

Sequential (4 ops)
Step  Operator                                                                  Input Signature                              Output Signature                           
----  ------------------------------------------------------------------------  -------------------------------------------  -------------------------------------------
0     ValidateCoords()                                                          (time=365, lat=181, lon=360); dtype=float32  (time=365, lat=181, lon=360); dtype=float32
1     SubsetBBox(lon_bnds=[-80, -50], lat_bnds=[30, 45], lon='lon', lat='lat')  (time=365, lat=181, lon=360); dtype=float32  (time=365, lat=?, lon=?); dtype=float32    
2     ResampleTime(freq='1D', method='mean', time='time')                       (time=365, lat=?, lon=?); dtype=float32      (time=?, lat=?, lon=?); dtype=float32      
3     Reduce(op='mean', dim=['lat', 'lon'], keepdims=False)                     (time=?, lat=?, lon=?); dtype=float32        (time=?); d

Reading the table:

- `ValidateCoords` is shape-preserving — no override needed; the default kicks in.
- `SubsetBBox` knows it touches `lat`/`lon` but can't tell *how many* points
  land inside the bbox without the actual coords; both axes go to `?`.
- `ResampleTime` similarly fuzzes `time`.
- `Reduce(dim=("lat","lon"))` drops the spatial axes; only `time` remains.

## 3. Sequential pipeline — climatology + anomaly setup

A different shape-changing pattern: `CalculateClimatology` swaps the time
axis for a `month` (or `dayofyear`/`year`) axis whose length depends on the
group-by — so it's `?` symbolically.

In [7]:
clim_pipe = Sequential([
    ValidateCoords(),
    SubsetTime(time_min="2000-01-01", time_max="2010-12-31"),
    CalculateClimatology(freq="month"),
])

print(clim_pipe.summary(Signature(
    {"time": 4017, "lat": 181, "lon": 360}, dtype="float32",
)))

Sequential (3 ops)
Step  Operator                                                               Input Signature                               Output Signature                            
----  ---------------------------------------------------------------------  --------------------------------------------  --------------------------------------------
0     ValidateCoords()                                                       (time=4017, lat=181, lon=360); dtype=float32  (time=4017, lat=181, lon=360); dtype=float32
1     SubsetTime(time_min='2000-01-01', time_max='2010-12-31', time='time')  (time=4017, lat=181, lon=360); dtype=float32  (time=?, lat=181, lon=360); dtype=float32   
2     CalculateClimatology(freq='month', time='time')                        (time=?, lat=181, lon=360); dtype=float32     (month=?, lat=181, lon=360); dtype=float32  


## 4. Sequential pipeline — coordinate reshape + regrid

This one mixes a CF rename, a coarsen, and a regrid onto another grid.
`RegridLike` knows the target's exact size, so the output is fully concrete.

In [8]:
target_grid = xr.Dataset(coords={
    "lat": np.linspace(-90, 90, 91),  # 2° lat
    "lon": np.linspace(0, 360, 180, endpoint=False),  # 2° lon
})

regrid_pipe = Sequential([
    RenameCoords({"longitude": "lon", "latitude": "lat"}),
    Coarsen({"lat": 2, "lon": 2}, boundary="trim"),
    RegridLike(target_grid),
])

print(regrid_pipe.summary(Signature(
    {"time": 12, "latitude": 720, "longitude": 1440}, dtype="float64",
)))

Sequential (3 ops)
Step  Operator                                                                                Input Signature                                         Output Signature                           
----  --------------------------------------------------------------------------------------  ------------------------------------------------------  -------------------------------------------
0     RenameCoords(mapping={'longitude': 'lon', 'latitude': 'lat'})                           (time=12, latitude=720, longitude=1440); dtype=float64  (time=12, lat=720, lon=1440); dtype=float64
1     Coarsen(factor={'lat': 2, 'lon': 2}, method='mean', boundary='trim')                    (time=12, lat=720, lon=1440); dtype=float64             (time=12, lat=360, lon=720); dtype=float64 
2     RegridLike(target_shape={'lat': 91, 'lon': 180}, dims=['lat', 'lon'], method='linear')  (time=12, lat=360, lon=720); dtype=float64              (time=12, lat=91, lon=180); dtype=float64  


Notice the rename flows through cleanly — the table tracks dim names, not
just sizes.

## 5. `Graph.summary` — multi-input metric

Sequential is for linear chains. The functional `Graph` API handles DAGs,
including ops with multiple inputs (the canonical case: pixel metrics like
RMSE that take `pred` and `ref`). The shape protocol propagates through:

In [9]:
pred = Input("pred")
ref = Input("ref")
score = RMSE(variable="ssh", dims=("time", "lat"))(pred, ref)
metric_graph = Graph(inputs={"pred": pred, "ref": ref}, outputs={"score": score})

# Both inputs share the same shape; metric reduces (time, lat) and keeps lon:
shape = Signature({"time": 365, "lat": 181, "lon": 360}, dtype="float32")
print(metric_graph.summary({"pred": shape, "ref": shape}))

Graph (2 inputs, 1 output)
Step  Operator                                    Input Signature                                                                           Output Signature        
----  ------------------------------------------  ----------------------------------------------------------------------------------------  ------------------------
2     RMSE(variable='ssh', dims=['time', 'lat'])  (time=365, lat=181, lon=360); dtype=float32, (time=365, lat=181, lon=360); dtype=float32  (lon=360); dtype=float64


`compute_output_signature` returns the same dict you'd get from running the
graph on real data — minus the data:

In [10]:
print(metric_graph.compute_output_signature({"pred": shape, "ref": shape}))

{'score': Signature(dims={'lon': 360}, dtype='float64')}


### What happens with mismatched shapes?

The pixel metric override validates dim names + sizes before reducing. If
you wire two inputs whose `time` length disagrees, `summary` fails loud —
you don't have to wait for an `xr.apply_ufunc` broadcast error at runtime.

In [11]:
mismatched = Signature({"time": 365, "lat": 181, "lon": 360}, dtype="float32")
shorter = Signature({"time": 90, "lat": 181, "lon": 360}, dtype="float32")

try:
    metric_graph.compute_output_signature({"pred": mismatched, "ref": shorter})
except ValueError as exc:
    print("caught:", exc)

caught: Metric input signature sizes do not match: {'time': (365, 90)}.


## 6. Single-input `Graph` (the convenience form)

Graphs with one input/output accept a bare `Signature` (you don't have to
wrap it in a dict). Same UX as `Sequential.summary`:

In [12]:
x = Input("ds")
y = SubsetBBox(lon_bnds=(0, 30), lat_bnds=(-30, 30))(x)
y = ResampleTime(freq="1MS")(y)
single_graph = Graph(inputs={"ds": x}, outputs={"out": y})

print(single_graph.summary(Signature(
    {"time": 365, "lat": 181, "lon": 360}, dtype="float32",
)))

Graph (1 input, 1 output)
Step  Operator                                                                Input Signature                              Output Signature                       
----  ----------------------------------------------------------------------  -------------------------------------------  ---------------------------------------
1     SubsetBBox(lon_bnds=[0, 30], lat_bnds=[-30, 30], lon='lon', lat='lat')  (time=365, lat=181, lon=360); dtype=float32  (time=365, lat=?, lon=?); dtype=float32
2     ResampleTime(freq='1MS', method='mean', time='time')                    (time=365, lat=?, lon=?); dtype=float32      (time=?, lat=?, lon=?); dtype=float32  


## 7. Nested `Graph`-in-`Graph` composition

Because `Graph` is itself an `Operator`, you can drop a fully-built Graph
into a larger Graph. Shape inference threads through every layer:

In [13]:
# Inner graph: time-subset + region crop on a single input.
inner_in = Input("ds_in")
cropped = SubsetTime(time_min="2010-01-01", time_max="2010-12-31")(inner_in)
cropped = SubsetBBox(lon_bnds=(-80, 0), lat_bnds=(20, 60))(cropped)
inner = Graph(inputs={"ds_in": inner_in}, outputs={"cropped": cropped})

# Outer graph: feed the input through the inner graph, then resample + reduce.
outer_in = Input("ds")
hooked = inner(outer_in)
resampled = ResampleTime(freq="1ME")(hooked)
reduced = Reduce(op="mean", dim=("lat", "lon"))(resampled)
outer = Graph(inputs={"ds": outer_in}, outputs={"global_mean": reduced})

print(outer.summary(Signature(
    {"time": 3650, "lat": 181, "lon": 360}, dtype="float32",
)))

Graph (1 input, 1 output)
Step  Operator                                               Input Signature                               Output Signature                     
----  -----------------------------------------------------  --------------------------------------------  -------------------------------------
1     Graph(inputs=['ds_in'], outputs=['cropped'])           (time=3650, lat=181, lon=360); dtype=float32  (time=?, lat=?, lon=?); dtype=float32
2     ResampleTime(freq='1ME', method='mean', time='time')   (time=?, lat=?, lon=?); dtype=float32         (time=?, lat=?, lon=?); dtype=float32
3     Reduce(op='mean', dim=['lat', 'lon'], keepdims=False)  (time=?, lat=?, lon=?); dtype=float32         (time=?); dtype=float32              


## 8. Common mistakes the summary catches early

- **`Coarsen` with `boundary="exact"` and a non-divisible factor.** `summary`
  refuses to render a misleading shape and points at the offending dim:

In [14]:
try:
    Sequential([Coarsen({"lat": 7}, boundary="exact")]).summary(
        Signature({"time": 12, "lat": 10, "lon": 8}, dtype="float32"),
    )
except ValueError as exc:
    print("caught:", exc)

caught: coarsen boundary='exact' requires 'lat' size 10 to be divisible by factor 7.


- **`Coarsen` with an unknown boundary string.** Constructor-time validation
  catches the typo before you ever call summary:

In [15]:
try:
    Coarsen({"lat": 2}, boundary="trip")
except ValueError as exc:
    print("caught:", exc)

caught: Coarsen boundary must be one of ('exact', 'trim', 'pad'), got 'trip'.


## 9. When to reach for which

- **`Sequential.describe()`** — structural tree only (operator names + configs).
  Doesn't need a `Signature`. Use when you want to show the *plan* of a
  pipeline.
- **`Sequential.summary(signature)`** — adds per-step input/output shape.
  Use when shape questions matter: "how many lat points after the regrid?",
  "where did the time axis go?".
- **`Graph.summary(...)`** — same idea for DAGs. Required when an operator
  has multiple inputs (metrics) or multiple outputs (multi-task pipelines).

All three are eager-free: no real data is touched, just signatures threaded
through `compute_output_signature` overrides.